In [ ]:
import os, time, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
from pathlib import Path
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.quantization import quantize_dynamic
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# Cache setup
CACHE_DIR     = Path('./cache'); CACHE_DIR.mkdir(exist_ok=True)
CACHE_SPEC    = CACHE_DIR / 'spectrograms.npz'
CACHE_SPLITS  = CACHE_DIR / 'splits.npz'
CACHE_HISTORY = CACHE_DIR / 'training_history.npz'
CACHE_EVAL    = CACHE_DIR / 'eval_results.npz'
CACHE_JAMMING = CACHE_DIR / 'jamming_results.npz'
MODEL_PATH    = Path('./best_sonar_convvit.pt')

def cache_status():
    files = {'Spectrograms':CACHE_SPEC,'Splits':CACHE_SPLITS,
             'Model':MODEL_PATH,'History':CACHE_HISTORY,
             'Eval':CACHE_EVAL,'Jamming':CACHE_JAMMING}
    print('\nCache status:')
    for label, path in files.items():
        if path.exists():
            print(f'  ✅ {label:15s} → {path.name} ({path.stat().st_size/1e6:.1f} MB)')
        else:
            print(f'  🔄 {label:15s} → will compute')

cache_status()

In [ ]:
# ── CHANGE THIS PATH TO YOUR FOLDER ─────────────────────────────────────────
SHIPSEAR_ROOT = Path('DS3500/ShipsEar_extracted/shipsear_5s_16k')
# ─────────────────────────────────────────────────────────────────────────────

NUM_CLASSES   = 5
CLASS_NAMES   = ['Motorboat','Fishing','Cargo/Tanker','Tugboat','Environment']
DEFENSE_LABEL = ['⚠ Hostile','👁 Monitor','✅ Non-Threat','👁 Monitor','🌊 Clear']

all_samples = []
for cid in range(NUM_CLASSES):
    cdir = SHIPSEAR_ROOT / str(cid)
    if not cdir.exists():
        print(f'WARNING: {cdir} not found'); continue
    for f in sorted(cdir.rglob('*.wav')):
        all_samples.append((str(f), cid))

label_counts = Counter(lbl for _, lbl in all_samples)
print(f'Total WAV files: {len(all_samples)}')
print(f'\n{"CID":>4} {"Class":>15} {"Count":>7} {"Pct":>7}  Defense')
print('-'*55)
for cid in range(NUM_CLASSES):
    n = label_counts[cid]
    print(f'{cid:>4} {CLASS_NAMES[cid]:>15} {n:>7} {n/len(all_samples)*100:>6.1f}%  {DEFENSE_LABEL[cid]}')

In [ ]:
SR, CLIP_DUR = 16000, 5
N_MELS, N_FFT, HOP = 128, 512, 256
IMG_H, IMG_W = 128, 304
PATCH_SIZE   = 16
pH = IMG_H // PATCH_SIZE   # 8
pW = IMG_W // PATCH_SIZE   # 19
NUM_PATCHES = pH * pW       # 152
print(f'Spectrogram : ({IMG_H}, {IMG_W})')
print(f'Patches     : {pH} x {pW} = {NUM_PATCHES} tokens')

def wav_to_raw_melspec(path):
    """WAV -> raw log-Mel dB. No normalization (done globally)."""
    y, _ = librosa.load(path, sr=SR, duration=CLIP_DUR, mono=True)
    target = SR * CLIP_DUR
    y = np.pad(y, (0, max(0, target - len(y))))[:target]
    mel     = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT,
                                              hop_length=HOP, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    w = log_mel.shape[1]
    if w < IMG_W: log_mel = np.pad(log_mel, ((0,0),(0, IMG_W-w)))
    else:         log_mel = log_mel[:, :IMG_W]
    return log_mel.astype(np.float32)

if CACHE_SPEC.exists():
    print('✅ Loading spectrograms from cache...')
    d           = np.load(CACHE_SPEC)
    all_specs   = d['specs']
    all_labels  = d['labels']
    GLOBAL_MEAN = float(d['global_mean'])
    GLOBAL_STD  = float(d['global_std'])
    print(f'   {len(all_specs)} specs | mean={GLOBAL_MEAN:.2f} dB | std={GLOBAL_STD:.2f} dB')
else:
    print(f'🔄 Converting {len(all_samples)} WAV files (runs ONCE)...')
    specs, labels, failed = [], [], []
    for path, lbl in tqdm(all_samples, desc='WAV→Mel'):
        try:
            specs.append(wav_to_raw_melspec(path))
            labels.append(lbl)
        except Exception as e:
            failed.append((path, str(e)))
    all_specs   = np.stack(specs).astype(np.float32)
    all_labels  = np.array(labels, dtype=np.int64)
    GLOBAL_MEAN = float(all_specs.mean())
    GLOBAL_STD  = float(all_specs.std())
    print(f'Global stats: mean={GLOBAL_MEAN:.2f} dB, std={GLOBAL_STD:.2f} dB')
    np.savez_compressed(CACHE_SPEC, specs=all_specs, labels=all_labels,
                        global_mean=GLOBAL_MEAN, global_std=GLOBAL_STD)
    print(f'✅ Saved → {CACHE_SPEC.name} ({CACHE_SPEC.stat().st_size/1e6:.1f} MB)')
    if failed: print(f'⚠  {len(failed)} files failed')

print(f'Specs  : {all_specs.shape}')
print(f'Labels : {all_labels.shape}')

In [ ]:
# Visualize one sample per class
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
fig.suptitle('Log-Mel Spectrograms — ShipsEar (from cache)', fontsize=12, fontweight='bold')
for cid, ax in enumerate(axes):
    idx      = np.where(all_labels == cid)[0][0]
    spec_vis = (all_specs[idx] - GLOBAL_MEAN) / (GLOBAL_STD + 1e-8)
    ax.imshow(spec_vis, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(f'Class {cid}: {CLASS_NAMES[cid]}\n{DEFENSE_LABEL[cid]}', fontsize=10)
    ax.set_xlabel('Time frames'); ax.set_ylabel('Mel bins')
plt.tight_layout()
plt.savefig('sonar_spectrograms.png', dpi=150)
plt.show()

In [ ]:
if CACHE_SPLITS.exists():
    print('✅ Loading splits from cache...')
    sp = np.load(CACHE_SPLITS)
    tr_idx, vl_idx, te_idx = sp['tr_idx'], sp['vl_idx'], sp['te_idx']
else:
    print('🔄 Computing stratified splits...')
    idx = np.arange(len(all_specs))
    tr_idx, tmp = train_test_split(idx, test_size=0.30,
                                    stratify=all_labels, random_state=SEED)
    vl_idx, te_idx = train_test_split(tmp, test_size=0.50,
                                       stratify=all_labels[tmp], random_state=SEED)
    np.savez(CACHE_SPLITS, tr_idx=tr_idx, vl_idx=vl_idx, te_idx=te_idx)
    print(f'✅ Saved → {CACHE_SPLITS.name}')

print(f'Train:{len(tr_idx)} | Val:{len(vl_idx)} | Test:{len(te_idx)}')

counts  = torch.tensor([label_counts[i] for i in range(NUM_CLASSES)], dtype=torch.float)
weights = (1.0/counts / (1.0/counts).sum() * NUM_CLASSES).to(DEVICE)
print(f'Class weights: {[f"{w:.2f}" for w in weights.cpu()]}')


class SonarDataset(Dataset):
    def __init__(self, specs, labels, indices, mean=0.0, std=1.0, augment=False):
        self.specs   = specs
        self.labels  = labels
        self.indices = indices
        self.mean    = mean
        self.std     = std
        self.augment = augment

    def __len__(self): return len(self.indices)

    def __getitem__(self, i):
        idx  = self.indices[i]
        spec = self.specs[idx].copy()
        lbl  = int(self.labels[idx])

        if self.augment:
            # 1. Time shift
            if random.random() < 0.5:
                spec = np.roll(spec, random.randint(-30, 30), axis=1)
            # 2. Frequency mask
            if random.random() < 0.5:
                f0 = random.randint(0, IMG_H - 20)
                fw = random.randint(5, 20)
                spec[f0:f0+fw, :] = self.mean
            # 3. Time mask
            if random.random() < 0.5:
                t0 = random.randint(0, IMG_W - 40)
                tw = random.randint(10, 40)
                spec[:, t0:t0+tw] = self.mean
            # 4. SNR noise (10-30 dB)
            if random.random() < 0.4:
                snr  = random.uniform(10, 30)
                spec = spec + (10**(-snr/20.0)) * np.random.randn(*spec.shape).astype(np.float32) * self.std
            # 5. Amplitude scale (distance simulation)
            if random.random() < 0.3:
                spec = spec * random.uniform(0.7, 1.3)

        # Global Z-score normalization
        spec = (spec - self.mean) / (self.std + 1e-8)
        return torch.tensor(spec, dtype=torch.float32).unsqueeze(0), lbl


BATCH    = 32
train_ds = SonarDataset(all_specs, all_labels, tr_idx, GLOBAL_MEAN, GLOBAL_STD, augment=True)
val_ds   = SonarDataset(all_specs, all_labels, vl_idx, GLOBAL_MEAN, GLOBAL_STD)
test_ds  = SonarDataset(all_specs, all_labels, te_idx, GLOBAL_MEAN, GLOBAL_STD)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=4, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=4, pin_memory=True)

x, y = next(iter(train_dl))
print(f'Batch shape : {x.shape} | min={x.min():.2f} max={x.max():.2f}')